# Data Preparation

Я нашел три датасета на kaggle по классификации фейков. Они все на английском, поэтому для поддержки русскуязычных статей будем использовать специально обученную для перевода новостей модель wmt19-ru-en.

Выбранные датасеты:
* https://www.kaggle.com/c/fake-news/data
* https://www.kaggle.com/c/fakenewskdd2020/data
* https://www.kaggle.com/c/classifying-the-fake-news/data

In [ ]:
import pandas as pd

df1_train = pd.read_csv('./data1/train.csv')

In [ ]:
df1_train

,id,title,author,text,label
0,0,House Dem Aide: We Didn’t Even See Comey’s Let...,Darrell Lucus,House Dem Aide: We Didn’t Even See Comey’s Let...,1
1,1,"FLYNN: Hillary Clinton, Big Woman on Campus - ...",Daniel J. Flynn,Ever get the feeling your life circles the rou...,0
2,2,Why the Truth Might Get You Fired,Consortiumnews.com,"Why the Truth Might Get You Fired October 29, ...",1
3,3,15 Civilians Killed In Single US Airstrike Hav...,Jessica Purkiss,Videos 15 Civilians Killed In Single US Airstr...,1
4,4,Iranian woman jailed for fictional unpublished...,Howard Portnoy,Print \nAn Iranian woman has been sentenced to...,1
...,...,...,...,...,...
20795,20795,Rapper T.I.: Trump a ’Poster Child For White S...,Jerome Hudson,Rapper T. I. unloaded on black celebrities who...,0
20796,20796,"N.F.L. Playoffs: Schedule, Matchups and Odds -...",Benjamin Hoffman,When the Green Bay Packers lost to the Washing...,0
20797,20797,Macy’s Is Said to Receive Takeover Approach by...,Michael J. de la Merced and Rachel Abrams,The Macy’s of today grew from the union of sev...,0
20798,20798,"NATO, Russia To Hold Parallel Exercises In Bal...",Alex Ansary,"NATO, Russia To Hold Parallel Exercises In Bal...",1


In [ ]:
df1_train['text'] = df1_train.apply(lambda x: str(x.title) + '. ' + str(x.text), axis=1)
df1_train = df1_train[['text', 'label']]

In [ ]:
df2_train = pd.read_csv('./data2/train.csv', sep='\t')

In [ ]:
# Битая строка
df2_train = df2_train.drop([1615])

In [ ]:
df2_train

,text,label
0,Get the latest from TODAY Sign up for our news...,1
1,2d Conan On The Funeral Trump Will Be Invited...,1
2,It’s safe to say that Instagram Stories has fa...,0
3,Much like a certain Amazon goddess with a lass...,0
4,At a time when the perfect outfit is just one ...,0
...,...,...
4982,The storybook romance of WWE stars John Cena a...,0
4983,The actor told friends he’s responsible for en...,0
4984,Sarah Hyland is getting real. The Modern Fami...,0
4985,Production has been suspended on the sixth and...,0


In [ ]:
df3_train = pd.read_csv('./data3/training.csv')

In [ ]:
df3_train['text'] = df3_train.apply(lambda x: str(x.title) + '. ' + str(x.text), axis=1)
df3_train = df3_train[['text', 'label']]

In [ ]:
all_data_train = df1_train.append(df2_train).append(df3_train)
all_data_train.to_csv('./train.csv', index=False)

# Training

In [ ]:
#!pip install transformers
import transformers

In [ ]:
from transformers import Trainer, TrainingArguments, LineByLineTextDataset

In [ ]:
import pandas as pd

In [ ]:
from datasets import Dataset

In [ ]:
df = pd.read_csv('./train.csv')

In [ ]:
df

,text,label
0,House Dem Aide: We Didn’t Even See Comey’s Let...,1
1,"FLYNN: Hillary Clinton, Big Woman on Campus - ...",0
2,Why the Truth Might Get You Fired.Why the Trut...,1
3,15 Civilians Killed In Single US Airstrike Hav...,1
4,Iranian woman jailed for fictional unpublished...,1
...,...,...
57209,CHICAGO TRUMP RALLY CANCELLED: Radicals And BL...,1
57210,Trump supports completion of Dakota Access Pip...,0
57211,Obama Can’t Stop Winning As New Jobs Report S...,1
57212,Turkey bank regulator dismisses 'rumors' after...,0


In [ ]:
df['labels'] = df['label']

In [ ]:
df = df[['text', 'labels']]

In [ ]:
dataset = Dataset.from_pandas(df)

In [ ]:
dataset

Dataset({
    features: ['text', 'labels'],
    num_rows: 57214
})

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel, pipeline

model_name = 'distilbert-base-uncased-finetuned-sst-2-english'
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
def preprocess_function(examples):
    return tokenizer(examples["text"], padding=True, truncation=True)

In [ ]:
dataset = dataset.map(preprocess_function, batched=True)

  0%|          | 0/58 [00:00<?, ?ba/s]

In [ ]:
dataset_splitted = dataset.shuffle(1337).train_test_split(0.1)

In [ ]:
dataset_splitted

DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'input_ids', 'attention_mask'],
        num_rows: 51492
    })
    test: Dataset({
        features: ['text', 'labels', 'input_ids', 'attention_mask'],
        num_rows: 5722
    })
})

In [ ]:
from transformers import AutoModelForSequenceClassification

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

loading configuration file https://huggingface.co/distilbert-base-uncased-finetuned-sst-2-english/resolve/main/config.json from cache at C:\Users\andry/.cache\huggingface\transformers\4e60bb8efad3d4b7dc9969bf204947c185166a0a3cf37ddb6f481a876a3777b5.9f8326d0b7697c7fd57366cdde57032f46bc10e37ae81cb7eb564d66d23ec96b
Model config DistilBertConfig {
  "_name_or_path": "distilbert-base-uncased-finetuned-sst-2-english",
  "activation": "gelu",
  "architectures": [
    "DistilBertForSequenceClassification"
  ],
  "attention_dropout": 0.1,
  "dim": 768,
  "dropout": 0.1,
  "finetuning_task": "sst-2",
  "hidden_dim": 3072,
  "id2label": {
    "0": "NEGATIVE",
    "1": "POSITIVE"
  },
  "initializer_range": 0.02,
  "label2id": {
    "NEGATIVE": 0,
    "POSITIVE": 1
  },
  "max_position_embeddings": 512,
  "model_type": "distilbert",
  "n_heads": 12,
  "n_layers": 6,
  "output_past": true,
  "pad_token_id": 0,
  "qa_dropout": 0.1,
  "seq_classif_dropout": 0.2,
  "sinusoidal_pos_embds": false,
  "ti

In [ ]:
for name, param in model.named_parameters():
    if name in ['classifier.weight', 'classifier.bias']:
        param.requires_grad = True
    else:
        param.requires_grad = False

In [ ]:
from sklearn.metrics import accuracy_score

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    acc = accuracy_score(labels, preds)
    return {'accuracy': acc}

In [ ]:
from transformers import Trainer, TrainingArguments

trainer = Trainer(
    model=model, train_dataset=dataset_splitted['train'],
    eval_dataset=dataset_splitted['test'],
    compute_metrics=compute_metrics,
    args=TrainingArguments(
        load_best_model_at_end=True,
        output_dir="./my_saved_model", overwrite_output_dir=True,
        num_train_epochs=10, per_device_train_batch_size=64,
        per_device_eval_batch_size=64,
        evaluation_strategy = "epoch",
        save_strategy = "epoch",
        save_steps=10_000, save_total_limit=2),
)

trainer.train()

PyTorch: setting up devices
The default value for the training argument `--report_to` will change in v5 (from all installed integrations to none). In v5, you will need to use `--report_to all` to get the same behavior as now. You should start updating your code and make this info disappear :-).
The following columns in the training set  don't have a corresponding argument in `DistilBertForSequenceClassification.forward` and have been ignored: text. If text are not expected by `DistilBertForSequenceClassification.forward`,  you can safely ignore this message.
***** Running training *****
  Num examples = 51492
  Num Epochs = 10
  Instantaneous batch size per device = 64
  Total train batch size (w. parallel, distributed & accumulation) = 64
  Gradient Accumulation steps = 1
  Total optimization steps = 8050


Epoch,Training Loss,Validation Loss,Accuracy
1,1.124500,0.655170,0.631423
2,0.635900,0.616928,0.696435
3,0.617400,0.592879,0.727019
4,0.591200,0.577941,0.734533
5,0.577100,0.564665,0.747466
6,0.569300,0.556096,0.749913
7,0.563200,0.551389,0.755330
8,0.559900,0.546756,0.754981
9,0.554800,0.544496,0.759000
10,0.554000,0.543604,0.760398


The following columns in the evaluation set  don't have a corresponding argument in `DistilBertForSequenceClassification.forward` and have been ignored: text. If text are not expected by `DistilBertForSequenceClassification.forward`,  you can safely ignore this message.
***** Running Evaluation *****
  Num examples = 5722
  Batch size = 64
Saving model checkpoint to ./my_saved_model\checkpoint-805
Configuration saved in ./my_saved_model\checkpoint-805\config.json
Model weights saved in ./my_saved_model\checkpoint-805\pytorch_model.bin
The following columns in the evaluation set  don't have a corresponding argument in `DistilBertForSequenceClassification.forward` and have been ignored: text. If text are not expected by `DistilBertForSequenceClassification.forward`,  you can safely ignore this message.
***** Running Evaluation *****
  Num examples = 5722
  Batch size = 64
Saving model checkpoint to ./my_saved_model\checkpoint-1610
Configuration saved in ./my_saved_model\checkpoint-1610\c

TrainOutput(global_step=8050, training_loss=0.6166538418598057, metrics={'train_runtime': 5516.6092, 'train_samples_per_second': 93.34, 'train_steps_per_second': 1.459, 'total_flos': 6.821011291594752e+16, 'train_loss': 0.6166538418598057, 'epoch': 10.0})